# Maia Lite 0.3.0 — SFT v2.2 LoRA (L4/A100)

Adaptado do piloto científico original, preservando arquitetura GPT, máscara de perda somente na resposta, adaptadores e retomada. **Execute a auditoria v2.2 antes.** Não execute treinamento até revisar as configurações.

In [ ]:
import os, json, math, time, random, hashlib, gc, csv
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer
from tqdm.auto import tqdm

SEED=20261118
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), 'Este piloto requer GPU CUDA; selecione A100 no Colab.'
print('GPU:', torch.cuda.get_device_name(0), '| Torch:',torch.__version__)
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
except ImportError:
    print('Fora do Colab: confirme os caminhos.')
assert Path('/content/drive/MyDrive').is_dir(), 'Drive não montado.'


GPU: NVIDIA L4 | Torch: 2.14.1+cu130


In [ ]:
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
PREP=ROOT/'datasets/sft_v2_2_aya'
TRAIN_FILE=PREP/'sft_v2_2_train.jsonl'
VAL_FILE=PREP/'sft_v2_2_validation.jsonl'
BASE_CKPT=ROOT/'training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt'
TOKENIZER_DIR=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
OUT=ROOT/'training_runs/maia_lite_0_3_0_sft_v2_2_lora_v1'
for path in (TRAIN_FILE,VAL_FILE,BASE_CKPT): assert path.is_file(),f'Arquivo ausente: {path}'
assert TOKENIZER_DIR.is_dir(),f'Tokenizer ausente: {TOKENIZER_DIR}'
for path in (TRAIN_FILE,VAL_FILE,BASE_CKPT): print(path, path.stat().st_size)
print('Saída:',OUT)


/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/datasets/sft_v2_2_aya/sft_v2_2_train.jsonl 30088710
/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/datasets/sft_v2_2_aya/sft_v2_2_validation.jsonl 1614568
/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt 6084855785
Saída: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_sft_v2_2_lora_v1


## Pré-requisito obrigatório: auditoria aprovada

Aprovação manual registrada no notebook de auditoria e hashes correspondentes são exigidos. Se qualquer arquivo mudar, refaça a auditoria.

In [ ]:
AUDIT=PREP/'audit_v2_2/audit_pass.json'
assert AUDIT.is_file(),f'Auditoria pendente: {AUDIT}'
audit=json.loads(AUDIT.read_text(encoding='utf8'))
def file_hash(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()
assert audit['issues_count']==0 and audit['human_review_approved'] and audit['license_review_approved']
assert audit['train_sha256']==file_hash(TRAIN_FILE),'Treino alterado após auditoria'
assert audit['validation_sha256']==file_hash(VAL_FILE),'Validação alterada após auditoria'
print('Auditoria aprovada e hashes verificados')

Auditoria aprovada e hashes verificados


## Configuração do piloto

Comece com **1 época**. Revise as métricas antes de alterar `EPOCHS` para 2 ou 3. A avaliação é feita por perda teacher-forced; respostas livres são testadas numa célula separada. `MAX_LENGTH=512` é um limite inicial de custo, menor que o contexto nativo de 2048; registros longos serão **excluídos**, nunca truncados silenciosamente. A implementação LoRA abaixo atua em `qkv` e `proj` de cada bloco.

In [ ]:
RANK=16; ALPHA=32; LORA_DROPOUT=0.05
LR=8e-5; WEIGHT_DECAY=0.0
MAX_LENGTH=512; EPOCHS=1; MICRO_BATCH=1; GRAD_ACCUM=16
MAX_GRAD_NORM=1.0; VALIDATE_EVERY=100; SAVE_EVERY=100
MAX_STEPS=None   # limite opcional para teste rápido (ex.: 10)
RUN_TRAINING=True  # ALTERE PARA True APÓS REVISAR A AUDITORIA E O PREVIEW
assert MAX_LENGTH<=2048


In [ ]:
from dataclasses import dataclass
@dataclass
class Config:
    vocab_size:int=50257
    block_size:int=2048
    n_layer:int=36
    n_head:int=16
    n_embd:int=1024
    dropout:float=0.0
    bias:bool=True
cfg=Config()
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        d=c.n_embd
        self.ln1=nn.LayerNorm(d)
        self.qkv=nn.Linear(d,3*d,bias=c.bias)
        self.proj=nn.Linear(d,d,bias=c.bias)
        self.ln2=nn.LayerNorm(d)
        self.fc=nn.Linear(d,4*d,bias=c.bias)
        self.out=nn.Linear(4*d,d,bias=c.bias)
        self.nh=c.n_head
    def forward(self,x):
        b,t,d=x.shape
        q,k,v=self.qkv(self.ln1(x)).chunk(3,dim=-1)
        q=q.view(b,t,self.nh,d//self.nh).transpose(1,2)
        k=k.view(b,t,self.nh,d//self.nh).transpose(1,2)
        v=v.view(b,t,self.nh,d//self.nh).transpose(1,2)
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0)
        x=x+self.proj(a.transpose(1,2).contiguous().view(b,t,d))
        x=x+self.out(F.gelu(self.fc(self.ln2(x)),approximate='tanh'))
        return x
class MaiaGPT(nn.Module):
    def __init__(self,c):
        super().__init__();self.cfg=c
        self.tok=nn.Embedding(c.vocab_size,c.n_embd)
        self.pos=nn.Embedding(c.block_size,c.n_embd)
        self.blocks=nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm=nn.LayerNorm(c.n_embd)
        self.head=nn.Linear(c.n_embd,c.vocab_size,bias=False)
        self.head.weight=self.tok.weight
        self.apply(self.init_weights)
    @staticmethod
    def init_weights(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,mean=0,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None:nn.init.zeros_(m.bias)
    def forward(self,idx,targets=None):
        b,t=idx.shape
        assert t<=self.cfg.block_size
        x=self.tok(idx)+self.pos(torch.arange(t,device=idx.device))
        for block in self.blocks:x=block(x)
        x=self.norm(x)
        logits=self.head(x)
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),targets.reshape(-1),ignore_index=-100)
        return logits,loss


## Carregar checkpoint original (somente leitura)

`weights_only=False` é necessário porque o checkpoint confiável do próprio treinamento contém estados RNG e optimizer. **Nunca use essa opção com checkpoints de origem desconhecida.** A arquitetura é reproduzida diretamente do notebook original v3.8.

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(str(TOKENIZER_DIR),local_files_only=True,use_fast=True)
assert len(tokenizer)==cfg.vocab_size,(len(tokenizer),cfg.vocab_size)
assert tokenizer.eos_token_id is not None,'Tokenizer sem EOS.'
print('EOS:',tokenizer.eos_token_id,'vocab:',len(tokenizer))
checkpoint=torch.load(BASE_CKPT,map_location='cpu',weights_only=False)
assert 'model' in checkpoint and 'step' in checkpoint, 'Formato de checkpoint inesperado.'
print('Step do checkpoint:',checkpoint['step'])
assert int(checkpoint['step'])==300000, 'Checkpoint não é step 300000: revise antes de continuar.'
model=MaiaGPT(cfg)
model.load_state_dict(checkpoint['model'],strict=True)
del checkpoint; gc.collect()
assert model.head.weight.data_ptr()==model.tok.weight.data_ptr()
model=model.to(device='cuda',dtype=torch.bfloat16)
for p in model.parameters():p.requires_grad_(False)
model.eval()
print('Modelo carregado, pesos base congelados.')


EOS: 2 vocab: 50257
Step do checkpoint: 300000
Modelo carregado, pesos base congelados.


In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, rank, alpha, dropout):
        super().__init__()
        assert isinstance(base,nn.Linear)
        self.base=base
        self.scale=alpha/rank
        self.dropout=nn.Dropout(dropout)
        self.A=nn.Parameter(torch.empty(rank,base.in_features,device=base.weight.device,dtype=torch.float32))
        self.B=nn.Parameter(torch.zeros(base.out_features,rank,device=base.weight.device,dtype=torch.float32))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
    def forward(self,x):
        # Base remains BF16; LoRA computation in BF16 autocast, parameters FP32.
        update=F.linear(F.linear(self.dropout(x).float(),self.A),self.B)*self.scale
        return self.base(x)+update.to(x.dtype)

for block in model.blocks:
    block.qkv=LoRALinear(block.qkv,RANK,ALPHA,LORA_DROPOUT)
    block.proj=LoRALinear(block.proj,RANK,ALPHA,LORA_DROPOUT)
trainable=[p for p in model.parameters() if p.requires_grad]
print('LoRA trainable parameters:',sum(p.numel() for p in trainable))
assert len(trainable)==cfg.n_layer*2*2


LoRA trainable parameters: 3538944


## Preparar sequências e máscara de perda

Formato fixo: `prompt` + `completion` + EOS. Apenas a resposta e o EOS recebem perda. A tokenização é feita sobre a sequência **completa**, para preservar merges BPE na fronteira prompt/resposta; a fronteira é estimada por `offset_mapping`. Amostras com fronteira ambígua são descartadas e relatadas.

In [ ]:
def read_jsonl(path):
    with open(path,encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]
train_raw=read_jsonl(TRAIN_FILE); val_raw=read_jsonl(VAL_FILE)
assert len(train_raw)>20000 and len(val_raw)>1000, (len(train_raw),len(val_raw))
assert not ({r['paperId'] for r in train_raw if r.get('paperId')}&{r['paperId'] for r in val_raw if r.get('paperId')})
assert not ({r['groupId'] for r in train_raw}&{r['groupId'] for r in val_raw})
print('Split sem sobreposição de paperId/groupId.')

def encode_record(rec):
    prompt=rec['prompt']; answer=rec['completion']
    if not prompt.strip() or not answer.strip():return None,'empty'
    text=prompt+answer
    enc=tokenizer(text,add_special_tokens=False,return_offsets_mapping=True)
    ids=enc['input_ids']; offsets=enc['offset_mapping']; boundary=len(prompt)
    # Reject BPE tokens that cross the prompt-answer boundary.
    if any(a<boundary<b for a,b in offsets):return None,'boundary_merge'
    answer_idx=next((i for i,(a,b) in enumerate(offsets) if a>=boundary and b>a),None)
    if answer_idx is None:return None,'no_answer_tokens'
    ids=ids+[tokenizer.eos_token_id]
    if len(ids)>MAX_LENGTH:return None,'too_long'
    # x=ids[:-1], labels predict ids[1:]; positions prior to answer are masked.
    labels=[-100]*(len(ids)-1)
    for i in range(len(labels)):
        if i+1>=answer_idx:labels[i]=ids[i+1]
    if all(v==-100 for v in labels):return None,'empty_targets'
    return {'x':ids[:-1],'y':labels,'language':rec['language'],'id':rec['id']},None

def prepare(rows):
    good=[]; dropped={}
    for r in rows:
        item,why=encode_record(r)
        if item is not None:good.append(item)
        else:dropped[why]=dropped.get(why,0)+1
    return good,dropped
train_data,train_drop=prepare(train_raw);val_data,val_drop=prepare(val_raw)
print('Train:',len(train_data),'excluídos:',train_drop)
print('Val:',len(val_data),'excluídos:',val_drop)
assert train_data and val_data
for d in (train_data,val_data):
    print('Idiomas:',{lang:sum(r['language']==lang for r in d) for lang in ['pt','en','es']})


Split sem sobreposição de paperId/groupId.


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1204 > 1024). Running this sequence through the model will result in indexing errors


Train: 28857 excluídos: {'too_long': 746, 'boundary_merge': 392}
Val: 1469 excluídos: {'boundary_merge': 20, 'too_long': 32}
Idiomas: {'pt': 6763, 'en': 14540, 'es': 7554}
Idiomas: {'pt': 100, 'en': 1017, 'es': 352}


In [ ]:
def collate(batch):
    L=max(len(r['x']) for r in batch)
    x=torch.full((len(batch),L),tokenizer.eos_token_id,dtype=torch.long)
    y=torch.full((len(batch),L),-100,dtype=torch.long)
    for i,r in enumerate(batch):
        l=len(r['x']);x[i,:l]=torch.tensor(r['x']);y[i,:l]=torch.tensor(r['y'])
    return x.to('cuda'),y.to('cuda')

sample=train_data[0]
print('Exemplo:',sample['id'],'| tokens:',len(sample['x']))
print('Tokens supervisionados:',sum(z!=-100 for z in sample['y']))
print('Resposta supervisionada (preview):',tokenizer.decode([z for z in sample['y'] if z!=-100])[:500])


Exemplo: W3020484714-q08::pt | tokens: 71
Tokens supervisionados: 24
Resposta supervisionada (preview): Controle das fontes de emissão, desenvolvimento de tecnologias de purificação do ar e otimização dos sistemas de ventilação</s>


## Avaliação e checkpoints LoRA

Os arquivos salvos contêm **somente os adaptadores**, optimizer, scheduler e RNG, nunca o checkpoint base. Retomada automática de época com ordem determinística por seed e época. **A retomada é por passo de atualização**, e pressupõe os mesmos dados e hiperparâmetros. O checkpoint é salvo por substituição atômica de arquivo local temporário no mesmo diretório Drive; confirme o arquivo após a execução.

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
DATA_HASHES={'train':sha256(TRAIN_FILE),'validation':sha256(VAL_FILE)}

def lora_state():
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items() if '.A' in k or '.B' in k}

def restore_lora(state):
    keys={k for k in model.state_dict() if '.A' in k or '.B' in k}
    assert set(state)==keys,(len(state),len(keys))
    with torch.no_grad():
        for name,param in model.named_parameters():
            if name in state:param.copy_(state[name].to(param.device,dtype=param.dtype))

@torch.no_grad()
def evaluate():
    model.eval(); sums={}; counts={}
    for lang in ['pt','en','es']:
        sums[lang]=0.;counts[lang]=0
    for start in range(0,len(val_data),MICRO_BATCH):
        batch=val_data[start:start+MICRO_BATCH]
        x,y=collate(batch)
        with torch.autocast('cuda',dtype=torch.bfloat16):logits,_=model(x)
        losses=F.cross_entropy(logits.float().reshape(-1,cfg.vocab_size),y.reshape(-1),ignore_index=-100,reduction='none').reshape_as(y)
        for i,r in enumerate(batch):
            mask=y[i]!=-100;lang=r['language']
            sums[lang]+=float(losses[i][mask].sum());counts[lang]+=int(mask.sum())
        del x,y,logits,losses
    result={f'val_loss_{k}':sums[k]/max(counts[k],1) for k in sums}
    result['val_loss']=sum(sums.values())/max(sum(counts.values()),1)
    model.train();return result

optimizer=torch.optim.AdamW(trainable,lr=LR,weight_decay=WEIGHT_DECAY)
updates_per_epoch=math.ceil(math.ceil(len(train_data)/MICRO_BATCH)/GRAD_ACCUM)
planned_updates=updates_per_epoch*EPOCHS
if MAX_STEPS is not None:planned_updates=min(planned_updates,MAX_STEPS)
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda s: max(0.1,1-s/max(planned_updates,1)))
print('Updates/epoch:',updates_per_epoch,'planned:',planned_updates)


Updates/epoch: 1804 planned: 1804


In [ ]:
OUT.mkdir(parents=True,exist_ok=True)
CONFIG={'rank':RANK,'alpha':ALPHA,'dropout':LORA_DROPOUT,'lr':LR,'max_length':MAX_LENGTH,
        'micro_batch':MICRO_BATCH,'grad_accum':GRAD_ACCUM,'epochs':EPOCHS,'max_steps':MAX_STEPS,
        'seed':SEED,'data_hashes':DATA_HASHES,'checkpoint_path':str(BASE_CKPT),'base_step':300000,
        'target_modules':['qkv','proj'],'architecture':'MaiaGPT v3.8'}
config_file=OUT/'config.json'
if config_file.exists():
    old=json.loads(config_file.read_text())
    assert old==CONFIG,'Configuração mudou em relação ao experimento existente. Use outro OUT.'
else:config_file.write_text(json.dumps(CONFIG,indent=2,ensure_ascii=False))

ckpt_file=OUT/'adapter_latest.pt';start_update=0
if ckpt_file.exists():
    saved=torch.load(ckpt_file,map_location='cpu',weights_only=False)
    assert saved['config']==CONFIG
    restore_lora(saved['adapter']);optimizer.load_state_dict(saved['optimizer']);scheduler.load_state_dict(saved['scheduler'])
    start_update=saved['update'];random.setstate(saved['python_rng']);np.random.set_state(saved['numpy_rng']);torch.set_rng_state(saved['torch_rng'])
    if torch.cuda.is_available():torch.cuda.set_rng_state_all(saved['cuda_rng'])
    print('Retomada do update:',start_update)
else:print('Novo experimento, nenhum adaptador anterior.')

def save_adapter(update,metrics=None):
    state={'adapter':lora_state(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
           'update':update,'config':CONFIG,'metrics':metrics,
           'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),'torch_rng':torch.get_rng_state(),
           'cuda_rng':torch.cuda.get_rng_state_all()}
    tmp=OUT/'adapter_latest.tmp.pt'
    torch.save(state,tmp);os.replace(tmp,ckpt_file)
    print('Adaptador salvo:',ckpt_file,'update:',update)

# Melhor validação conhecida, persistida para retomadas
BEST_META=OUT/'best_validation.json'
best_loss=float(json.loads(BEST_META.read_text())['val_loss']) if BEST_META.exists() else float('inf')
def maybe_save_best(update,metrics):
    global best_loss
    if metrics and 'val_loss' in metrics and metrics['val_loss']<best_loss:
        best_loss=float(metrics['val_loss'])
        state={'adapter':lora_state(),'update':update,'val_loss':best_loss,'config':CONFIG}
        temp=OUT/'adapter_best.tmp.pt'
        torch.save(state,temp);os.replace(temp,OUT/'adapter_best.pt')
        BEST_META.write_text(json.dumps({'update':update,'val_loss':best_loss},indent=2))
        print('Melhor adaptador:',update,best_loss)


Novo experimento, nenhum adaptador anterior.


## Executar piloto

Primeiro rode com `RUN_TRAINING=False` para verificar a preparação e a avaliação inicial. Depois altere para `True` e execute a célula abaixo. Não altere hiperparâmetros no meio de uma retomada. O loop calcula a perda ponderada por **tokens supervisionados** (não pela quantidade de padding).

In [ ]:
if not RUN_TRAINING:
    print('Treinamento desativado. Pré-avaliação:',evaluate())
else:
    assert start_update<=planned_updates
    model.train(); optimizer.zero_grad(set_to_none=True)
    update=start_update
    progress=tqdm(total=planned_updates,initial=update,desc='SFT LoRA')
    log_path=OUT/'metrics.jsonl'
    # Deterministic epoch permutations and batch boundaries for restart at update boundaries.
    try:
        for epoch in range(EPOCHS):
            if update>=planned_updates:break
            order=np.random.default_rng(SEED+epoch).permutation(len(train_data)).tolist()
            micro_batches=[order[i:i+MICRO_BATCH] for i in range(0,len(order),MICRO_BATCH)]
            n_updates=math.ceil(len(micro_batches)/GRAD_ACCUM)
            for local_update in range(n_updates):
                global_update=epoch*updates_per_epoch+local_update
                if global_update<update:continue
                if update>=planned_updates:break
                indices=micro_batches[local_update*GRAD_ACCUM:(local_update+1)*GRAD_ACCUM]
                # Normalize accumulated token losses by the total supervised tokens in the update.
                total_tokens=sum(sum(z!=-100 for z in train_data[j]['y']) for ids in indices for j in ids)
                accumulated_loss=0.
                for ids in indices:
                    x,y=collate([train_data[j] for j in ids])
                    with torch.autocast('cuda',dtype=torch.bfloat16):logits,_=model(x)
                    loss_sum=F.cross_entropy(logits.float().reshape(-1,cfg.vocab_size),y.reshape(-1),ignore_index=-100,reduction='sum')
                    (loss_sum/max(total_tokens,1)).backward()
                    accumulated_loss+=float(loss_sum.detach())
                    del x,y,logits,loss_sum
                grad_norm=torch.nn.utils.clip_grad_norm_(trainable,MAX_GRAD_NORM)
                optimizer.step();scheduler.step();optimizer.zero_grad(set_to_none=True)
                update+=1;progress.update(1)
                metrics={'update':update,'epoch':epoch+1,'train_loss':accumulated_loss/max(total_tokens,1),
                         'grad_norm':float(grad_norm),'lr':optimizer.param_groups[0]['lr']}
                if update%VALIDATE_EVERY==0 or update==planned_updates:
                    metrics.update(evaluate());print(metrics);maybe_save_best(update,metrics)
                with open(log_path,'a',encoding='utf-8') as f:f.write(json.dumps(metrics)+'\n')
                if update%SAVE_EVERY==0 or update==planned_updates:save_adapter(update,metrics)
    except KeyboardInterrupt:
        optimizer.zero_grad(set_to_none=True)
        print('Interrupção: somente atualizações completas são retomáveis. Último update:',update)
        save_adapter(update)
        raise
    finally:progress.close()
    print('Treino concluído até update',update)


SFT LoRA:   0%|          | 0/1804 [00:00<?, ?it/s]

{'update': 100, 'epoch': 1, 'train_loss': 2.4169732948090723, 'grad_norm': 0.6262159943580627, 'lr': 7.556541019955654e-05, 'val_loss_pt': 2.8573514698252414, 'val_loss_en': 2.5644583896960205, 'val_loss_es': 2.2693441390014284, 'val_loss': 2.4733631398956994}
Melhor adaptador: 100 2.4733631398956994
Adaptador salvo: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_sft_v2_2_lora_v1/adapter_latest.pt update: 100
{'update': 200, 'epoch': 1, 'train_loss': 2.5597753553604625, 'grad_norm': 0.6151902079582214, 'lr': 7.113082039911308e-05, 'val_loss_pt': 2.837294468253902, 'val_loss_en': 2.4981757843293844, 'val_loss_es': 2.251084984836215, 'val_loss': 2.4245650482153387}
Melhor adaptador: 200 2.4245650482153387
Adaptador salvo: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_sft_v2_2_lora_v1/adapter_latest.pt update: 200
{'update': 300, 'epoch': 1, 'train_loss': 1.8240903642226238, 'grad_norm': 0.5744302868843079, 'lr':

## Teste qualitativo de geração (antes/depois)

O modelo base não possui chat template. A geração usa o mesmo estilo de prompt do SFT e é **apenas diagnóstica**, não substitui avaliação independente. Para comparar base e adaptador, execute com LoRA ativo e, separadamente, desative os módulos LoRA ou carregue o modelo base em uma nova sessão.

In [ ]:
@torch.no_grad()
def generate(prompt,max_new_tokens=100,temperature=0.7):
    model.eval()
    ids=tokenizer(prompt,add_special_tokens=False,return_tensors='pt')['input_ids'].to('cuda')
    for _ in range(max_new_tokens):
        if ids.shape[1]>=cfg.block_size:break
        with torch.autocast('cuda',dtype=torch.bfloat16):logits,_=model(ids)
        last=logits[:,-1,:].float()/max(temperature,1e-4)
        nxt=torch.multinomial(torch.softmax(last,dim=-1),1)
        if nxt.item()==tokenizer.eos_token_id:break
        ids=torch.cat([ids,nxt],dim=1)
    model.train()
    return tokenizer.decode(ids[0],skip_special_tokens=True)

for lang in ['pt','en','es']:
    r=next(r for r in val_raw if r['language']==lang)
    print('\n',lang,r['id'],'\n',generate(r['prompt'])[-1200:])



 pt W2906764419-q02::pt 
 Responda à pergunta de forma clara, objetiva e cientificamente correta. Se não souber, indique a incerteza. Quais são as duas técnicas exatas de decomposição celular mencionadas no artigo?As técnicas de decomposição celular listadas são a hora de dissecação e a hora de separação de células.

 en oasst:0d988311-a34b-4ae4-9fd0-2b9aaa3d62e5 
 Answer clearly, concisely, and accurately. If unsure, acknowledge uncertainty. How is the use of Chinese Characters in Japanese different than the use in the Chinese languages?Chinese characters are generally used in the language of Japan. However, there is a difference in the way they are used in the different languages. In the Chinese language, there are different types of Chinese characters, such as: 1. 段 (比) (王) 2. 於 (新) (川) 3. 敷 (悲) 4. 教 (根) 5. 悲 (

 es oasst:4341a788-438c-4482-b28a-95c223bb9f33 
 Responde de forma clara, concisa y precisa. Si no lo sabes, indica la incertidumbre. ¿Cuales son las ventajas que trae el p

## Próximos passos

Verifique `metrics.jsonl`, `best_validation.json`, a perda por idioma e respostas livres. Compare com o modelo-base e o SFT científico v1 usando **benchmark externo**. O melhor adaptador fica em `adapter_best.pt`; o último estado retomável em `adapter_latest.pt`. Não confunda menor perda teacher-forced com qualidade conversacional garantida.

In [6]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat")
PREP = ROOT / "datasets/sft_v2_2_aya"

print("Google Drive montado:", Path("/content/drive/MyDrive").is_dir())
print("Pasta do projeto:", ROOT.is_dir())
print("Pasta do dataset:", PREP.is_dir())

if PREP.is_dir():
    print("\nArquivos disponíveis:")
    for p in sorted(PREP.iterdir()):
        print(p.name)

Google Drive montado: False
Pasta do projeto: False
Pasta do dataset: False


In [7]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [8]:
from pathlib import Path
import json

VAL_FILE = Path(
    "/content/drive/MyDrive/Colab_LLMs/"
    "Maia_Lite_0_3_0_Chat/datasets/sft_v2_2_aya/"
    "sft_v2_2_validation.jsonl"
)

assert VAL_FILE.is_file(), f"Arquivo não encontrado: {VAL_FILE}"

with VAL_FILE.open(encoding="utf-8") as f:
    val_raw = [json.loads(line) for line in f if line.strip()]

print(f"Validação carregada: {len(val_raw)} exemplos")

Validação carregada: 1521 exemplos


In [9]:
import json

with open(VAL_FILE, "r", encoding="utf-8") as f:
    val_raw = [json.loads(line) for line in f if line.strip()]

print(f"Validação carregada: {len(val_raw)} exemplos")

Validação carregada: 1521 exemplos


In [11]:
# ============================================================
# MAIA LITE 0.3.0 — INFERÊNCIA APÓS REINICIAR O COLAB
# Base: checkpoint 300000
# LoRA: SFT v2.2 — melhor adaptador
# Não executa treinamento
# ============================================================

import sys
import subprocess
import json
import math
import gc
from pathlib import Path
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from transformers import AutoTokenizer
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip",
        "install", "-q", "transformers"
    ])
    from transformers import AutoTokenizer

from google.colab import drive

# 1. GPU e Google Drive
assert torch.cuda.is_available(), "Selecione uma GPU no Colab."

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

device = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

# 2. Caminhos originais do projeto
ROOT = Path(
    "/content/drive/MyDrive/Colab_LLMs/"
    "Maia_Lite_0_3_0_Chat"
)

BASE_CKPT = (
    ROOT /
    "training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt"
)

TOKENIZER_DIR = Path(
    "/content/drive/MyDrive/Colab_LLMs/"
    "Maia_Lite_v3/tokenizer"
)

OUT = ROOT / "training_runs/maia_lite_0_3_0_sft_v2_2_lora_v1"

ADAPTER = OUT / "adapter_best.pt"

DATA_DIR = ROOT / "datasets/sft_v2_2_aya"
VAL_FILE = DATA_DIR / "sft_v2_2_validation.jsonl"

for path in [BASE_CKPT, TOKENIZER_DIR, ADAPTER]:
    assert path.exists(), f"Arquivo/diretório ausente: {path}"

print("Checkpoint base:", BASE_CKPT)
print("Adaptador:", ADAPTER)

# 3. Arquitetura original MaiaGPT
@dataclass
class Config:
    vocab_size: int = 50257
    block_size: int = 2048
    n_layer: int = 36
    n_head: int = 16
    n_embd: int = 1024
    dropout: float = 0.0
    bias: bool = True

cfg = Config()

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        d = c.n_embd
        self.ln1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3*d, bias=c.bias)
        self.proj = nn.Linear(d, d, bias=c.bias)
        self.ln2 = nn.LayerNorm(d)
        self.fc = nn.Linear(d, 4*d, bias=c.bias)
        self.out = nn.Linear(4*d, d, bias=c.bias)
        self.nh = c.n_head

    def forward(self, x):
        b, t, d = x.shape
        q, k, v = self.qkv(self.ln1(x)).chunk(3, dim=-1)
        q = q.view(b,t,self.nh,d//self.nh).transpose(1,2)
        k = k.view(b,t,self.nh,d//self.nh).transpose(1,2)
        v = v.view(b,t,self.nh,d//self.nh).transpose(1,2)

        a = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=0.0
        )
        x = x + self.proj(
            a.transpose(1,2).contiguous().view(b,t,d)
        )
        x = x + self.out(
            F.gelu(self.fc(self.ln2(x)), approximate="tanh")
        )
        return x

class MaiaGPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.cfg = c
        self.tok = nn.Embedding(c.vocab_size, c.n_embd)
        self.pos = nn.Embedding(c.block_size, c.n_embd)
        self.blocks = nn.ModuleList(
            [Block(c) for _ in range(c.n_layer)]
        )
        self.norm = nn.LayerNorm(c.n_embd)
        self.head = nn.Linear(
            c.n_embd, c.vocab_size, bias=False
        )
        self.head.weight = self.tok.weight

    def forward(self, idx, targets=None):
        b, t = idx.shape
        assert t <= self.cfg.block_size
        x = self.tok(idx) + self.pos(
            torch.arange(t, device=idx.device)
        )
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)).float(),
                targets.reshape(-1),
                ignore_index=-100
            )
        return logits, loss

# 4. Implementação LoRA original
class LoRALinear(nn.Module):
    def __init__(self, base, rank, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / rank
        self.dropout = nn.Dropout(dropout)

        self.A = nn.Parameter(torch.empty(
            rank, base.in_features,
            device=base.weight.device,
            dtype=torch.float32
        ))
        self.B = nn.Parameter(torch.zeros(
            base.out_features, rank,
            device=base.weight.device,
            dtype=torch.float32
        ))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        update = F.linear(
            F.linear(self.dropout(x).float(), self.A),
            self.B
        ) * self.scale
        return self.base(x) + update.to(x.dtype)

# 5. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    str(TOKENIZER_DIR),
    local_files_only=True,
    use_fast=True
)
assert len(tokenizer) == cfg.vocab_size
assert tokenizer.eos_token_id is not None

# 6. Carregar checkpoint original
# Somente checkpoint próprio e confiável
print("\nCarregando modelo-base...")

checkpoint = torch.load(
    BASE_CKPT,
    map_location="cpu",
    weights_only=False
)

assert int(checkpoint["step"]) == 300000

model = MaiaGPT(cfg)
model.load_state_dict(checkpoint["model"], strict=True)
del checkpoint
gc.collect()

model = model.to(device=device, dtype=torch.bfloat16)

for p in model.parameters():
    p.requires_grad_(False)

print("Modelo-base carregado.")

# 7. Ler configuração e adaptador LoRA
print("Carregando adaptador LoRA...")

saved = torch.load(
    ADAPTER,
    map_location="cpu",
    weights_only=False
)

lora_config = saved["config"]

RANK = int(lora_config["rank"])
ALPHA = int(lora_config["alpha"])
DROPOUT = float(lora_config["dropout"])

for block in model.blocks:
    block.qkv = LoRALinear(
        block.qkv, RANK, ALPHA, DROPOUT
    )
    block.proj = LoRALinear(
        block.proj, RANK, ALPHA, DROPOUT
    )

adapter_state = saved["adapter"]

expected = {
    name for name, _ in model.named_parameters()
    if name.endswith(".A") or name.endswith(".B")
}

assert set(adapter_state) == expected, (
    "Incompatibilidade entre adaptador e arquitetura LoRA."
)

with torch.no_grad():
    for name, param in model.named_parameters():
        if name in adapter_state:
            param.copy_(
                adapter_state[name].to(
                    device=param.device,
                    dtype=param.dtype
                )
            )

model.eval()

print("Adaptador carregado.")
print("Passo LoRA:", saved.get("update"))
print("Melhor validation loss:", saved.get("val_loss"))

del saved, adapter_state
gc.collect()

# 8. Geração gulosa
@torch.inference_mode()
def generate_greedy(prompt, max_new_tokens=120):
    model.eval()

    ids = tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors="pt"
    )["input_ids"].to(device)

    generated = []

    for _ in range(max_new_tokens):
        if ids.shape[1] >= cfg.block_size:
            break

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16
        ):
            logits, _ = model(ids)

        next_id = torch.argmax(
            logits[:, -1, :],
            dim=-1,
            keepdim=True
        )

        token_id = int(next_id.item())

        if token_id == tokenizer.eos_token_id:
            break

        generated.append(token_id)
        ids = torch.cat([ids, next_id], dim=1)

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

# 9. Três perguntas fixas, sem depender do JSONL
tests = {
    "pt": (
        "Responda à pergunta de forma clara, objetiva e "
        "cientificamente correta. Se não souber, indique "
        "a incerteza. "
        "Quais são os principais benefícios do plantio "
        "de árvores nas cidades?"
    ),
    "en": (
        "Answer clearly, concisely, and accurately. "
        "If unsure, acknowledge uncertainty. "
        "How is the use of Chinese characters in Japanese "
        "different from their use in Chinese languages?"
    ),
    "es": (
        "Responde de forma clara, concisa y precisa. "
        "Si no lo sabes, indica la incertidumbre. "
        "¿Cuáles son las ventajas de plantar árboles "
        "para la población?"
    )
}

print("\n" + "="*65)
print("MAIA LITE 0.3.0 — TESTE GULOSO APÓS SFT v2.2")
print("="*65)

for language, prompt in tests.items():
    print(f"\nIDIOMA: {language.upper()}")
    print("PERGUNTA:", prompt)
    print("RESPOSTA:")
    print(generate_greedy(prompt))
    print("-"*65)

print("\nTeste concluído. Nenhum treinamento foi executado.")

GPU: NVIDIA L4
Checkpoint base: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt
Adaptador: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_sft_v2_2_lora_v1/adapter_best.pt

Carregando modelo-base...
Modelo-base carregado.
Carregando adaptador LoRA...
Adaptador carregado.
Passo LoRA: 1800
Melhor validation loss: 2.319600362225873

MAIA LITE 0.3.0 — TESTE GULOSO APÓS SFT v2.2

IDIOMA: PT
PERGUNTA: Responda à pergunta de forma clara, objetiva e cientificamente correta. Se não souber, indique a incerteza. Quais são os principais benefícios do plantio de árvores nas cidades?
RESPOSTA:
O plantio de árvores nas cidades é uma maneira de reduzir a poluição e melhorar a qualidade de vida.
-----------------------------------------------------------------

IDIOMA: EN
PERGUNTA: Answer clearly, concisely, and accurately. If unsure, acknowledge uncertainty. How is the use of Chinese characters in 